In [1]:
import polars as pl
import os
import glob
import gc
import gzip
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURATION
# ==========================================
INPUT_ROOT = r"D:\MASTER THESIS DATA\bitmex\gz" 
OUTPUT_ROOT = r"D:\MASTER THESIS DATA\bitmex\processed_5m_final" 

# Set this to 0 to run all
START_FROM_INDEX = 0

def process_daily_files_final():
    if not os.path.exists(OUTPUT_ROOT):
        os.makedirs(OUTPUT_ROOT)

    print(f"Scanning {INPUT_ROOT}...")
    all_files = glob.glob(os.path.join(INPUT_ROOT, "*.csv.gz"))
    all_files.sort()
    
    total_files = len(all_files)
    print(f"Found {total_files} daily files.")
    print(f"Starting from index: {START_FROM_INDEX}")

    # Strict Schema
    schema_map = {
        "bidPrice": pl.Float64,
        "askPrice": pl.Float64
    }

    for i, file_path in enumerate(all_files):
        if i < START_FROM_INDEX: continue

        try:
            filename = os.path.basename(file_path)
            date_str = filename.split('.')[0] 
            
            print(f"[{i}/{total_files}] Processing {filename}...", end=" ", flush=True)

            with gzip.open(file_path, 'rb') as f:
                df_raw = pl.read_csv(
                    f, 
                    columns=["timestamp", "symbol", "bidPrice", "askPrice"],
                    dtypes=schema_map
                )

            q = df_raw.lazy()

            # 1. FILTER SYMBOLS & PARSE TIMESTAMP
            q = (
                q
                .filter(
                    (~pl.col("symbol").str.contains("_")) &
                    (~pl.col("symbol").str.contains(r"\d$"))
                )
                .with_columns([
                    pl.col("timestamp")
                      .str.replace("D", " ")
                      .str.slice(0, 19) 
                      .str.to_datetime(format="%Y-%m-%d %H:%M:%S")
                ])
            )

            # 2. CALCULATE METADATA & FLAG VALIDITY
            # We preserve everything, but we tag the "Garbage"
            q = q.with_columns([
                
                # --- THE COMPREHENSIVE CHECK ---
                (
                    (pl.col("bidPrice").is_not_null()) & 
                    (pl.col("askPrice").is_not_null()) & 
                    (pl.col("bidPrice") > 0) & 
                    (pl.col("askPrice") > 0) &
                    (pl.col("askPrice") >= pl.col("bidPrice")) # Ask must be >= Bid
                ).alias("is_valid"),
                
                # Calculate Mid Price (Even if invalid, calculate it so we have something)
                ((pl.col("bidPrice") + pl.col("askPrice")) / 2).alias("mid_price")
            ])
            
            # 3. RESAMPLE TO 5 MIN (LAST STATE)
            # We capture the state of the market at the closing second of the bar.
            # If the market was "Crossed" (Broken) at 12:04:59, the bar is marked Invalid.
            q_resampled = (
                q
                .set_sorted("timestamp") 
                .group_by_dynamic("timestamp", every="5m", group_by="symbol")
                .agg([
                    pl.col("bidPrice").last().alias("bid_price"),
                    pl.col("askPrice").last().alias("ask_price"),
                    pl.col("mid_price").last().alias("mid_price"),
                    pl.col("is_valid").last().alias("is_valid")
                ])
            )

            # 4. COLLECT
            df_day = q_resampled.collect()

            if df_day.height == 0:
                print("No valid perps.")
                del df_raw, q, q_resampled, df_day
                gc.collect()
                continue

            # 5. PARTITION & SAVE
            unique_symbols = df_day["symbol"].unique().to_list()
            
            for sym in unique_symbols:
                df_sym = df_day.filter(pl.col("symbol") == sym)
                
                sym_folder = os.path.join(OUTPUT_ROOT, sym)
                os.makedirs(sym_folder, exist_ok=True)
                
                out_file = os.path.join(sym_folder, f"{date_str}.parquet")
                df_sym.write_parquet(out_file)

            print(f"Done. ({len(unique_symbols)} symbols)")

            del df_raw, df_day, q, q_resampled
            gc.collect()

        except Exception as e:
            print(f"\nError: {e}")
            gc.collect()

    print("\nETL Complete.")

process_daily_files_final()

Scanning D:\MASTER THESIS DATA\bitmex\gz...
Found 1034 daily files.
Starting from index: 0
[0/1034] Processing 20230101.csv.gz... Done. (41 symbols)
[1/1034] Processing 20230102.csv.gz... Done. (41 symbols)
[2/1034] Processing 20230103.csv.gz... Done. (41 symbols)
[3/1034] Processing 20230104.csv.gz... Done. (41 symbols)
[4/1034] Processing 20230105.csv.gz... Done. (41 symbols)
[5/1034] Processing 20230106.csv.gz... Done. (41 symbols)
[6/1034] Processing 20230107.csv.gz... Done. (41 symbols)
[7/1034] Processing 20230108.csv.gz... Done. (41 symbols)
[8/1034] Processing 20230109.csv.gz... Done. (41 symbols)
[9/1034] Processing 20230110.csv.gz... Done. (41 symbols)
[10/1034] Processing 20230111.csv.gz... Done. (43 symbols)
[11/1034] Processing 20230112.csv.gz... Done. (43 symbols)
[12/1034] Processing 20230113.csv.gz... Done. (43 symbols)
[13/1034] Processing 20230114.csv.gz... Done. (43 symbols)
[14/1034] Processing 20230115.csv.gz... Done. (43 symbols)
[15/1034] Processing 20230116.csv.